In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *
from cavity_correction import correct_cavity
from prefilter_correction import correct_prefilter

In [2]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/'
flat_folders = sorted(glob.glob(folder + '*'))

print(flat_folders)
flat_folder = flat_folders[-7]
flat_files = sorted(glob.glob(flat_folder + '/*.fits'))

['/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-03', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-06', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-15', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-10-11', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-03-30', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-09-26', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-10-16', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-10-27', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-12-02', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-01-19', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-03-10', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-09-15', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-09-23', '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2026-03-10', '/home/ulyanov/data/solo/phi/flat

In [3]:
flat_files = sorted(glob.glob('../process/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('../process/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('../process/temp/*cavity*.fits'))

print(flat_files)

['../process/temp/phi-fdt-flat_20240330T050009_V202608181121C_0463300100.fits', '../process/temp/phi-fdt-flat_20250310T080009_V202608181157C_0563100100.fits']


In [4]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'

i = 1

cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(cavity_file) as hdul:
    cavity = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data
    header = hdul[0].header

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

#ghost = demodulate(ghost, header)
flat_ = demodulate(flat, header)
flat_[1:] /= flat_[0]
flat_ -= np.mean(flat_, axis=(-2,-1), keepdims=True)

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(cavity, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[3], 'gray', vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [7]:
plt.figure(figsize=(10,10))
plt.imshow(flat[0], 'gray', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [8]:
def calc_ghost_scaling(data, header, sigma=0.043, gamma=0.053, depth=0.66, cavity=0):
    from scipy.signal import convolve

    xr, yr = reflection_point_predict(header)
    wv = read_wavelengths(header)
    cpos = header['CONTPOS'] - 1
    wv = wv - np.delete(wv, cpos)[2]

    shift = get_wv_shift(data, header)
    shift = reflect(shift + cavity, xr, yr) - cavity

    dx = 0.001
    x = np.arange(-10,10 + dx / 2, dx)
    f = 1 - depth * np.exp(-x ** 2 / 2 / sigma ** 2)
    g = 1 / (1 + (x / gamma) ** 2)
    q = 2 * (1 - convolve(f, g ** 2, mode='same') / convolve(f, g, mode='same'))

    Q = interpolate(q.reshape(-1,1,1), x, wv.reshape(-1,1,1) - np.expand_dims(shift, axis=0))
    return Q

In [76]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'
#folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080009_V202503131733C_0563100100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080608_V202503131733C_0563100125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081208_V202503131835C_0563100150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081808_V202503131935C_0563100175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T082408_V202503131935C_0563100200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083008_V202503132033C_0563100225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083608_V202503141634C_0563100250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [91]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

xr, yr = reflection_point_predict(header)
wv = read_wavelengths(header)
cpos = header['CONTPOS'] - 1
print(cpos)

nx, ny = data.shape[-2:]

data = data.reshape(6,4,nx,ny)
data -= crop(dark, header) * 0.4
data = correct_prefilter(data, header, prefilter_file)
data /= crop(flat, header)

data = correct_cavity(data, header, cavity)
#data = realign(data)

5


In [11]:
wv_shift = get_wv_shift(data, header)

In [12]:
plt.figure(figsize=(10,10))
plt.imshow(wv_shift, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [87]:
#from scipy.special import voigt_profile

def voigt_profile(z, sigma, gamma, q=1):
    if isinstance(z, np.ndarray):
        return np.array([voigt_profile(z_, sigma, gamma, q) for z_ in z])
    else:
        dx = 0.001
        x = np.arange(-10,10 + dx / 2, dx)
        f = np.exp(-x ** 2 / 2 / sigma ** 2)
        g = 1 / (1 + ((z - x) / gamma) ** 2)
        return np.trapezoid(f * g ** q, x)


sigma = 0.043
gamma = 0.053
contpos = cpos
acc = 1e-3
lam = 1e-6

n_wv = len(wv)

wv_min = np.min(np.delete(wv, contpos) if contpos is not None else wv)
wv_max = np.max(np.delete(wv, contpos) if contpos is not None else wv)
wvc = (wv_min + wv_max) / 2
wv_ = np.arange(wv_min, wv_max + acc / 2, acc, dtype=np.float32)
n_wv_ = len(wv_)

xi = np.expand_dims(wv, axis=1) - np.expand_dims(wv_, axis=0)
A = voigt_profile(xi, sigma, gamma)
A0 = np.mean(A, axis=0, keepdims=True)
A -= A0

M = np.zeros((n_wv, n_wv))
N = np.zeros((n_wv, n_wv))

for k in range(n_wv_):
    gk = voigt_profile(wv_[k] - wvc, sigma, gamma) ** 2

    for j in range(n_wv):
        mjk = gk * (2 * (voigt_profile(wv_[k] - wv[j], sigma, gamma) - voigt_profile(wv_[k] - wv[j], sigma, gamma, q=2)) - A0[0,k])
        njk = gk * A[j,k]

        for i in range(n_wv):
            M[j, i] += mjk * A[i,k]
            N[j, i] += njk * A[i,k]

Q = M @ np.linalg.inv(N + lam * np.identity(n_wv))
Q = Q @ (np.identity(n_wv) - 1 / n_wv) + 1 / n_wv

In [79]:
Q

array([[ 0.12209431,  0.35908496,  0.06946583,  0.01081998, -0.01448678,
         0.4530217 ],
       [ 0.46782591, -0.17404627,  0.47122205,  0.00557208,  0.03153289,
         0.19789333],
       [ 0.0481047 ,  0.45455462, -0.1703012 ,  0.45727893,  0.02506561,
         0.18529734],
       [ 0.05366809,  0.0062574 ,  0.46678916, -0.16407644,  0.43458558,
         0.20277621],
       [ 0.03895516,  0.00855377,  0.063872  ,  0.37487486,  0.0545071 ,
         0.45923712],
       [ 0.03069691, -0.01809114,  0.03909987, -0.02324684,  0.13627312,
         0.83526808]])

In [92]:
#q = calc_ghost_scaling(data, header)#, cavity=cavity)
reflection = reflect(data[:,0], xr, yr)
reflection = np.matmul(Q, reflection, axes=[(-2, -1), (0, 1), (0, 1)])
#reflection = correct_cavity(reflection, header, -cavity)
reflection = gaussian_filter(reflection, 8, axes=(-2,-1))
temp = data - np.expand_dims(reflection, 1) * np.expand_dims(crop(ghost, header), 0)
temp = demodulate(temp, header)

In [97]:
i = 2
j = 1

a, b = np.nanpercentile(temp[i,0], 0.1), np.nanpercentile(temp[i,0], 99.9)

plt.figure(figsize=(10,10))
plt.imshow(temp[i,j], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a))#, origin='lower')
plt.tight_layout()

In [25]:
plt.figure(figsize=(10,10))
plt.imshow(reflection[1] - reflection[5])
plt.tight_layout()